# LARSA 4D Macros

A first attempt at defining a simple two-span beam in LARSA 4D through the direct `LarsaElements` COM library (Approach 1 from the supplied examples).

**Environment:** this approach requires Windows, 32-bit Python, the LARSA 4D COM libraries, and `pywin32`. The support and load values below are illustrative only, not design recommendations. Review them and the output in LARSA before using this model for any real structure.

The example creates two 120-inch spans, each divided into ten 12-inch beam elements. Exactly three joints have support restraints (at x = 0, 120, and 240 inches); the other 18 joints are completely free. Later cells apply illustrative self-weight and uniform loading, save an `.lar` file, and run a staged analysis.

## Connect to LARSA and set project units

This follows the direct-library setup used in `LARSA4D_Macro_Approach1.py`. Adjust the units or coordinates below if the intended model uses a different unit system.

In [1]:
import win32com.client
from win32com.client import gencache

# Generate the type-library constants, then keep element objects late-bound.
gencache.EnsureDispatch("LarsaData.clsUnits")
gencache.EnsureDispatch("LarsaElements.clsProject")
gencache.GetClassForCLSID = lambda clsid: None

def CreateObject(class_name):
    return win32com.client.Dispatch(class_name)

project = CreateObject("LarsaElements.clsProject")

# Set all nine project unit categories to inches, kips, Fahrenheit, and radians.
for category in range(9):
    project.units.SetUnits(
        category,
        True,
        project.units.UnitFromString(win32com.client.constants.UNIT_LENGTH, "in"),
        project.units.UnitFromString(win32com.client.constants.UNIT_FORCE, "kip"),
        project.units.UnitFromString(win32com.client.constants.UNIT_TEMPERATURE, "F"),
        project.units.UnitFromString(win32com.client.constants.UNIT_ANGLE, "rad"),
    )

## Define material and section

These are illustrative values adapted from the supplied macro example. The LARSA COM material exposes `modulusOfElasticity` and `shearModulus`, but no Poisson-ratio property. For isotropic steel, calculate shear modulus as `G = E / (2 * (1 + nu))`. The section below is a custom property-defined section, attached at both member ends with `Setsection`; replace its properties with those for the actual cross-section and unit system.

In [2]:
material = CreateObject("LarsaElements.clsMaterial")
material.Name = "Steel"
material.modulusOfElasticity = 29000.0
poisson_ratio = 0.30
material.shearModulus = material.modulusOfElasticity / (2.0 * (1.0 + poisson_ratio))
material.unitWeight = 0.000284
project.materials.append(material)

section = CreateObject("LarsaElements.clsSection")
section.Name = "Illustrative beam section"
section.sectionArea = 10.0
section.inertiaY = 100.0
section.inertiaZ = 100.0
section.torsionj = 50.0
project.sections.append(section)

## Create joints and members

Each 120-inch span is divided into ten 12-inch elements. Supports are at x = 0, 120, and 240 inches; all intermediate joints have all six degrees of freedom free.

In [3]:
span_length = 120.0
elements_per_span = 10
element_length = span_length / elements_per_span
joints = []
for joint_index in range(2 * elements_per_span + 1):
    x = joint_index * element_length
    joint = CreateObject("LarsaElements.clsJoint")
    joint.location.setCoordinates(x, 0, 0)
    joint.constraint = "000000"  # Initialize every joint free; only three are restrained below.
    project.joints.append(joint)
    joints.append(joint)

members = []
for start_joint, end_joint in zip(joints, joints[1:]):
    member = CreateObject("LarsaElements.clsMember")
    member.Setjoint(1, start_joint)
    member.Setjoint(2, end_joint)
    member.material = material
    member.Setsection(1, section)
    member.Setsection(2, section)
    start_section_id = member.section(1)[0].number
    end_section_id = member.section(2)[0].number
    if start_section_id != section.number or end_section_id != section.number:
        raise RuntimeError(
            f"Member section assignment failed: expected {section.number}, "
            f"got start={start_section_id}, end={end_section_id}"
        )
    project.members.append(member)
    members.append(member)

expected_member_count = 2 * elements_per_span
if project.members.Count != expected_member_count:
    raise RuntimeError(f"Expected {expected_member_count} members, got {project.members.Count}")
print(f"Created {project.joints.Count} joints and {project.members.Count} members ({elements_per_span} per span).")
print(f"Steel E = {material.modulusOfElasticity:g} kip/in^2; G = {material.shearModulus:.2f} kip/in^2.")
print(f"Assigned custom section {section.Name!r} (ID {section.number}) to all members.")

Created 21 joints and 20 members (10 per span).
Steel E = 29000 kip/in^2; G = 11153.85 kip/in^2.
Assigned custom section 'Illustrative beam section' (ID 1) to all members.


## Model check and next steps

The intended geometry is a straight, 240-inch line with 20 twelve-inch members (ten per span) and exactly three restrained joints at x = 0, 120, and 240 inches. The other 18 joints have all six degrees of freedom free. The final code cells apply example supports and loads, save `TwoSpanBeam_10_elements_3_supports.lar` in the notebook's current working directory, and invoke the LARSA engine. These assumptions demonstrate the API only: check the resulting model, units, section, supports, and load interpretation in LARSA 4D before relying on it.

## Add illustrative supports and load cases

Only joints 1, 11, and 21 receive restraints: the first support fixes translations and torsion, and the middle and end supports restrain the two transverse translations. Every other joint remains fully free. This is an illustrative continuous two-span beam, not a verified design. The uniform load is 0.1 kip/in downward in global Z.

In [4]:
# DOF order: tx, ty, tz, rx, ry, rz; 1 = fixed, 0 = free.
for joint in joints:
    joint.constraint = "000000"  # All six DOFs free.
joints[0].constraint = "111100"  # Pin translations and restrain torsion at the first support.
joints[elements_per_span].constraint = "011000"  # Central support.
joints[-1].constraint = "011000"  # End support.

self_weight_case = CreateObject("LarsaElements.clsLoadCase")
self_weight_case.Name = "Illustrative self-weight"
project.primaryLoadCases.append(self_weight_case)
self_weight_case.weightFactor.z = -1

uniform_load_case = CreateObject("LarsaElements.clsLoadCase")
uniform_load_case.Name = "Illustrative uniform load"
project.primaryLoadCases.append(uniform_load_case)

for member in members:
    member_load = CreateObject("LarsaElements.clsMemberLoad")
    uniform_load_case.MemberLoads.append(member_load)
    member_load.member = member
    member_load.loadType = win32com.client.constants.UniformForce
    member_load.loadDir = win32com.client.constants.Global_z
    member_load.startW = -0.1

group = CreateObject("LarsaElements.clsGeoGroup")
group.Name = "Illustrative two-span beam"
project.geoGroups.append(group)
for member in members:
    group.addObject(win32com.client.constants.GGA_MEMBERS, member.number)

print("Applied illustrative supports, self-weight, and uniform load.")

Applied illustrative supports, self-weight, and uniform load.


## Save and attempt staged analysis

This follows the staged-analysis and engine invocation pattern in the supplied Approach 1 example. The output path is printed so you can locate the saved model. The engine can exit successfully even if its own analysis log reports input errors, so this cell checks that log before result retrieval.

In [5]:
import subprocess
from pathlib import Path

stage = CreateObject("LarsaElements.clsStage")
stage.Name = "Illustrative staged analysis"
project.stages.append(stage)

construction_step = CreateObject("LarsaElements.clsStageStep")
stage.steps.append(construction_step)
construction_step.Name = "Construction and self-weight"
construction_step.stageType = win32com.client.constants.ST_CONSTRUCTION
construction_step.constructionMethod = win32com.client.constants.SCM_STANDARD
construction_step.analysisMethod = win32com.client.constants.AT_NONLINEAR_STATIC
construction_step.groupAdd(group)
construction_step.caseAdd(self_weight_case, 1.0)

loading_step = CreateObject("LarsaElements.clsStageStep")
stage.steps.append(loading_step)
loading_step.Name = "Uniform loading"
loading_step.stageType = win32com.client.constants.ST_CONSTRUCTION
loading_step.constructionMethod = win32com.client.constants.SCM_STANDARD
loading_step.analysisMethod = win32com.client.constants.AT_NONLINEAR_STATIC
loading_step.caseAdd(uniform_load_case, 1.0)

project.analysisType = win32com.client.constants.AT_STAGE_STANDARD
project.analysisStageStart = 1
project.analysisStageEnd = project.stages.Count

output_path = Path.cwd() / "TwoSpanBeam_10_elements_3_supports.lar"
lar_format_version = (8, 9, 0)
project.SaveToFile(
    str(output_path),
    win32com.client.constants.FILE_FORMAT_LAR6A,
    *lar_format_version,
    None,
)
print(f"Saved model: {output_path}")

engine_exe = r"C:\Program Files (x86)\Larsa 2000\LarsaEngine64.exe"
subprocess.run([engine_exe, "/run", str(output_path)], check=True)
analysis_log_path = Path.cwd() / "LarsaResults" / output_path.stem / "output.log"
if not analysis_log_path.is_file():
    raise FileNotFoundError(f"LARSA analysis log was not created: {analysis_log_path}")
analysis_log = analysis_log_path.read_text(encoding="latin-1", errors="replace").replace("\x00", "")
analysis_errors = [line.strip() for line in analysis_log.splitlines() if "ERROR>" in line]
if analysis_errors:
    raise RuntimeError("LARSA reported analysis errors:\n" + "\n".join(analysis_errors))
print("LARSA analysis completed; the engine log contains no ERROR entries.")

Saved model: C:\Users\ant.glotov\OneDrive - BYCN\Desktop\LARSA Training\larsa4d-api-libraries\Python\LARSA 4D Macros\TwoSpanBeam_10_elements_3_supports.lar
LARSA analysis completed; the engine log contains no ERROR entries.


## Reload and verify an analysis result

Reload the saved project, verify it contains 20 members, and check vertical displacement at the free joint halfway through the first span (x = 60 inches) for the uniform-loading stage. Also retrieve a member sectional-force result. The six force components are left in the LARSA API's order; consult the LARSA result headers/documentation for labels and sign conventions.

In [6]:
result_project = CreateObject("LarsaElements.clsProject")
analysis = CreateObject("LarsaElements.clsAnalysisResults")
analysis.Load(result_project)
result_project.ReadFromFile(str(output_path), analysis, False)
analysis.Load(result_project)

result_case_name = "Illustrative staged analysis: Uniform loading"
matching_cases = [case for case in analysis.GetAllCases(None) if case.Name == result_case_name]
if not matching_cases:
    available_cases = [case.Name for case in analysis.GetAllCases(None)]
    raise RuntimeError(f"Analysis result case not found: {result_case_name!r}; available: {available_cases}")

result_case = matching_cases[0]
displacement_type = win32com.client.constants.RESULTDATA_JOINT_DISPLACEMENTS
joint_indexes = [0] * 4
midspan_joint_id = elements_per_span // 2 + 1
joint_indexes[1] = analysis.getJointIndexFromID(midspan_joint_id)[0]
raw_displacement = analysis.loadData3(
    result_case, displacement_type, 0, False, False, False, 0, [], [], joint_indexes
)[8]
converted_displacement = analysis.convertData(
    displacement_type, [0] + list(raw_displacement), result_project.units
)[1][1:]
joint_dofs = [float(value) for value in converted_displacement[2:]]
midspan_z_displacement = joint_dofs[2]
if abs(midspan_z_displacement) <= 1e-8:
    raise RuntimeError(f"Expected nonzero vertical displacement at free joint {midspan_joint_id}; got {midspan_z_displacement}")

data_type = win32com.client.constants.RESULTDATA_MEMBER_SECTIONAL_FORCES
indexes = [0] * 4
indexes[1] = analysis.getMemberIndexFromID(1)[0]
indexes[2] = 0
indexes[3] = 1
raw_data = analysis.loadData3(
    result_case, data_type, 0, False, False, False, 0, [], [], indexes
)[8]
converted_data = analysis.convertData(data_type, [0] + list(raw_data), result_project.units)[1][1:]
force_components = list(converted_data[3:])
if not any(isinstance(value, (int, float)) and abs(value) > 1e-12 for value in force_components):
    raise RuntimeError(f"Expected a nonzero member-force result; got {force_components}")

if result_project.joints.Count != 2 * elements_per_span + 1 or result_project.members.Count != expected_member_count:
    raise RuntimeError(f"Unexpected reloaded geometry: {result_project.joints.Count} joints, {result_project.members.Count} members")
print(f"Reloaded model: {result_project.joints.Count} joints, {result_project.members.Count} members.")
print(f"Free joint {midspan_joint_id} at x={element_length * (midspan_joint_id - 1):g} in vertical displacement: {midspan_z_displacement:.6g} in.")
print(f"Result case: {result_case_name}")
print(f"Member 1, station 0 force components (LARSA API order): {force_components}")

Reloaded model: 21 joints, 20 members.
Free joint 6 at x=60 in vertical displacement: -0.038299 in.
Result case: Illustrative staged analysis: Uniform loading
Member 1, station 0 force components (LARSA API order): [0.0, 0.0, 4.627799987792969, 0.0, -2.704295121169764e-13, 0.0]
